In [2]:
import pandas as pd
import numpy as np
import os
import joblib
import warnings
from collections import Counter

# Machine Learning Imports
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings('ignore')

# --- 1. Load Data ---
print("🔽 Loading dataset...")
try:
    df = pd.read_csv('synthetic_reports_egypt.csv')
except FileNotFoundError:
    print("❌ Error: 'synthetic_reports_egypt.csv' not found. Please run the generation script first.")
    exit()

# =============================================================================
# COLUMNS REMOVED AND WHY:
# ❌ proximity_to_service  → Data Leakage: assigned directly from report_truth
# ❌ repetition_flag       → Data Leakage: assigned directly from report_truth
# ❌ latitude / longitude  → Misleading: most governorates return (30.0, 31.0) constant
# ❌ text_length           → Misleading: length is a byproduct of label type
# ❌ report_date           → Random: no real temporal pattern in synthetic data
# ❌ day_of_week / month   → Derived from random date, zero predictive value
# =============================================================================

# --- 2. Feature Engineering (Preprocessing) ---

# Combine headline and details into a single rich text feature
df['text_features'] = df['report_headline'].fillna('') + ' ' + df['report_detail'].fillna('')

# Define column groups — only genuine, leak-free features
text_col         = 'text_features'
categorical_cols = ['issue_type', 'governorate', 'city', 'report_category']

# Detect language profile of the text to decide stop_words strategy
sample_text = ' '.join(df['text_features'].dropna().head(50).tolist())
arabic_chars = sum(1 for c in sample_text if '\u0600' <= c <= '\u06ff')
latin_chars  = sum(1 for c in sample_text if c.isascii() and c.isalpha())
is_arabic_dominant = arabic_chars > latin_chars

if is_arabic_dominant:
    print("🌍 Detected Arabic/mixed text — disabling English stop words")
    tfidf_stop_words = None
    tfidf_analyzer   = 'char_wb'   # character n-grams work better for Arabic
else:
    print("🌍 Detected English text — using English stop words")
    tfidf_stop_words = 'english'
    tfidf_analyzer   = 'word'

# Prepare X and y
X = df[categorical_cols + [text_col]]
y = df['report_truth']

# Encode target labels
label_encoder = LabelEncoder()
y_encoded     = label_encoder.fit_transform(y)
class_names   = label_encoder.classes_

print(f"📌 Classes: {class_names}")
print(f"📌 Features used: {categorical_cols + [text_col]}")

# Split — stratify to preserve class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# --- 3. Build the Pipeline ---
print("\n🛠️ Constructing the Production Pipeline...")

preprocessor = ColumnTransformer(
    transformers=[
        # TF-IDF on combined text (headline + detail)
        ('text', TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            stop_words=tfidf_stop_words,
            analyzer=tfidf_analyzer
        ), text_col),

        # One-Hot Encoding for categorical features
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ]
)

# --- Compute XGBoost class weights manually ---
# VotingClassifier passes sample_weight but XGBClassifier needs it explicitly;
# scale_pos_weight only applies to binary, so we use sample_weight at fit time.
# The compute_sample_weight call below handles this for all classifiers uniformly.

# Base Models
clf1 = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
clf2 = RandomForestClassifier(class_weight='balanced', n_estimators=100, n_jobs=-1, random_state=42)
clf3 = XGBClassifier(
    eval_metric='mlogloss',
    random_state=42,
    use_label_encoder=False
    # sample_weight is passed via fit — this is the correct multi-class approach
)

# Ensemble (soft voting for probability-based threshold tuning)
ensemble_model = VotingClassifier(
    estimators=[
        ('lr',  clf1),
        ('rf',  clf2),
        ('xgb', clf3)
    ],
    voting='soft'
)

# Full Pipeline
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   ensemble_model)
])

# --- 4. Cross-Validation (before final fit) ---
print("\n🔁 Running 5-Fold Stratified Cross-Validation...")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    full_pipeline, X, y_encoded,
    cv=cv, scoring='f1_macro', n_jobs=-1
)
print(f"✅ CV Macro F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"   Per-fold scores: {[f'{s:.4f}' for s in cv_scores]}")

# --- 5. Training (final fit on full train split) ---
print("\n🚀 Training the Ensemble Model (LR + RF + XGB)...")

# Sample weights handle class imbalance for ALL three classifiers uniformly
weights = compute_sample_weight(class_weight='balanced', y=y_train)
full_pipeline.fit(X_train, y_train, classifier__sample_weight=weights)

# --- 6. Evaluation ---
print("\n📊 Evaluating performance (Threshold = 0.5)...")
y_probs = full_pipeline.predict_proba(X_test)
y_pred  = full_pipeline.predict(X_test)

print(f"✅ Accuracy:       {accuracy_score(y_test, y_pred):.4f}")
print(f"✅ Macro F1-Score: {f1_score(y_test, y_pred, average='macro'):.4f}")
print("\n--- Detailed Classification Report ---")
print(classification_report(y_test, y_pred, target_names=class_names))

# --- 7. Emergency Threshold Tuning ---
# Lower threshold → higher recall for Emergency (safety-critical class)

def sensitive_predict(probabilities, target_idx, threshold=0.3):
    """
    Flags a report as Emergency if its probability >= threshold.
    For all other classes, argmax is computed over the remaining
    probabilities (Emergency excluded) to avoid double-counting.
    """
    predictions = []
    for prob in probabilities:
        if prob[target_idx] >= threshold:
            predictions.append(target_idx)
        else:
            remaining       = prob.copy()
            remaining[target_idx] = 0          # exclude Emergency from argmax
            predictions.append(int(np.argmax(remaining)))
    return np.array(predictions)

emergency_idx  = list(class_names).index('Emergency')
y_pred_tuned   = sensitive_predict(y_probs, target_idx=emergency_idx, threshold=0.3)

print("\n⚠️  Performance after Tuning Emergency Threshold (Threshold = 0.3):")
print(classification_report(y_test, y_pred_tuned, target_names=class_names))

# --- 8. Save Pipeline & Metadata ---
print("\n💾 Saving the Full Pipeline, Label Encoder & Metadata...")
model_dir = 'model_components'
os.makedirs(model_dir, exist_ok=True)

joblib.dump(full_pipeline, os.path.join(model_dir, 'full_classification_pipeline.pkl'))
joblib.dump(label_encoder, os.path.join(model_dir, 'label_encoder.pkl'))

# Save all inference-time settings in one place so inference scripts
# never have hardcoded thresholds or feature lists
metadata = {
    'emergency_threshold': 0.3,
    'emergency_idx':       emergency_idx,
    'class_names':         list(class_names),
    'categorical_cols':    categorical_cols,
    'text_col':            text_col,
    'tfidf_analyzer':      tfidf_analyzer,
    'tfidf_stop_words':    tfidf_stop_words,
    'cv_macro_f1_mean':    float(cv_scores.mean()),
    'cv_macro_f1_std':     float(cv_scores.std()),
}
joblib.dump(metadata, os.path.join(model_dir, 'model_metadata.pkl'))
print(f"✅ Metadata saved: {metadata}")

print("\n✅ Training complete. Clean pipeline saved successfully!")
print("\n📝 Note: Accuracy may be lower than before — that's EXPECTED and CORRECT.")
print("   The previous high accuracy was inflated by data leakage features.")
print("   This result reflects real model performance on genuine text signals.")

🔽 Loading dataset...
🌍 Detected English text — using English stop words
📌 Classes: ['Emergency' 'False' 'Out_of_scope' 'Repeated' 'True' 'Unclear']
📌 Features used: ['issue_type', 'governorate', 'city', 'report_category', 'text_features']

🛠️ Constructing the Production Pipeline...

🔁 Running 5-Fold Stratified Cross-Validation...
✅ CV Macro F1: 0.9881 ± 0.0023
   Per-fold scores: ['0.9843', '0.9894', '0.9881', '0.9875', '0.9913']

🚀 Training the Ensemble Model (LR + RF + XGB)...

📊 Evaluating performance (Threshold = 0.5)...
✅ Accuracy:       0.9840
✅ Macro F1-Score: 0.9877

--- Detailed Classification Report ---
              precision    recall  f1-score   support

   Emergency       0.99      0.91      0.95       461
       False       1.00      1.00      1.00       293
Out_of_scope       1.00      1.00      1.00       294
    Repeated       1.00      1.00      1.00       461
        True       0.97      0.99      0.98      1189
     Unclear       1.00      1.00      1.00       302
